# Spirit Board Example Notebook

This notebook demonstrates how to use the `spirit_board` Python library to communicate with the Spirit Board invoker firmware.

The Spirit Board is a hardware interface that allows you to control Furby toys by communicating with their proprietary serial bus.

## Quick Start

1. Update the `ESP32_HOST` variable below with your Spirit Board's IP address
2. Run the cells to connect and test basic functionality
3. Explore the predefined Furby commands
4. Try motor control and custom command sequences

## Setup

Import the library and configure the connection.

In [ ]:
import sys
import time

# Add software directory to path so we can import spirit_board
sys.path.insert(0, '../software')

from spirit_board import SpiritBoard, parse_nibble_echo, get_furby_command, list_furby_commands

In [ ]:
# Configure your Spirit Board IP address here
ESP32_HOST = "192.168.66.133"  # UPDATE THIS with your Spirit Board's IP address

# Create the Spirit Board connection
board = SpiritBoard(host=ESP32_HOST)

print(f"Spirit Board configured at {ESP32_HOST}:{board.port}")

## Basic Connectivity

Test the connection to the Spirit Board firmware.

In [ ]:
# Ping the firmware to verify connectivity
response = board.ping()
print("Ping response:", response)

# Should return '31337' if working correctly
if response == '31337':
    print("Connection successful!")
else:
    print("Warning: Unexpected ping response")

## Initialize the Furby Coprocessor

Before sending commands, you should initialize the Furby coprocessor by pulsing the /INIT line.

In [ ]:
# Initialize the Furby coprocessor
response = board.init_coprocessor()
print("Init response:", response)

## Send the Initialization Sequence

The initialization sequence prepares the Furby to receive commands.

In [ ]:
# Send the standard initialization sequence
init_cmd = get_furby_command("init")
print(f"Sending init sequence ({len(init_cmd)} hex chars)...")

response = board.send_nibble_stream(init_cmd)
print("\nResponse:")
print(response)

## List Available Furby Commands

The library includes several predefined Furby command sequences.

In [ ]:
# List all available commands
commands = list_furby_commands()
print("Available Furby commands:")
for cmd in commands:
    print(f"  - {cmd}")

## Execute a Furby Command

Let's make the Furby say "mee-mee"!

In [ ]:
# Make sure the coprocessor is initialized first
board.init_coprocessor()
time.sleep(0.1)

# Send the initialization sequence
board.send_nibble_stream(get_furby_command("init"))
time.sleep(0.1)

# Now send the "mee-mee" command
response = board.send_nibble_stream(get_furby_command("mee-mee"))
print("Command sent!")
print("Response:", response[:200], "...")  # Print first 200 chars

# Now send the "mee-mee" command
response = board.send_nibble_stream(get_furby_command("sequence-4"))
print("Command sent!")
print("Response:", response[:200], "...")  # Print first 200 chars

In [ ]:
sequence-1 is kinda corrupted with some humming
sequence-2 is totally corrupted
sequence-3 is some kind of growl




## Parse Nibble Echo

The firmware echoes back the nibbles it received. We can parse this response.

In [ ]:
# Send a command and parse the nibble echo
response = board.send_nibble_stream(get_furby_command("giggle"))

try:
    count, nibbles = parse_nibble_echo(response)
    print(f"Received {count} nibbles")
    print(f"First 20 nibbles: {nibbles[:20]}")
    
    # Convert back to hex string
    hex_str = ''.join(f"{n:X}" for n in nibbles[:20])
    print(f"As hex string: {hex_str}")
except ValueError as e:
    print(f"Could not parse nibbles: {e}")

## Get RTS Timing Data

After sending a nibble stream, you can retrieve detailed timing information about the /RTS signal.

In [ ]:
# Get RTS timing data from the last transmission
timing_response = board.get_rts_timing()
print("RTS Timing Response:")
print(timing_response[:500], "...")  # Print first 500 chars

## Motor Control

The Spirit Board can also control the Furby motor directly.

In [ ]:
# Drive the motor forward for 200 milliseconds (200,000 microseconds)
print("Driving motor forward for 200ms...")
response = board.motor_forward(200_000)
print("Response:", response)

# Wait a moment
time.sleep(0.5)

# Drive the motor backward for 200 milliseconds
print("\nDriving motor backward for 200ms...")
response = board.motor_backward(200_000)
print("Response:", response)

## Send a Custom Hex Sequence

You can also send your own custom hex sequences.

In [ ]:
# Send a custom sequence (this is just an example - replace with your own)
custom_sequence = "F7F3 47F7 F71C 0000"

# Whitespace in the hex string is automatically stripped
response = board.send_nibble_stream(custom_sequence)
print("Response:", response)

## Complete Example: Run Multiple Commands

Here's a complete workflow to initialize the Furby and run several commands.

In [ ]:
def run_furby_sequence(board, command_names, init_delay=0.5, cmd_delay=1.0):
    """
    Run a sequence of Furby commands with proper initialization.
    
    Args:
        board: SpiritBoard instance
        command_names: List of command names to execute
        init_delay: Delay after initialization (seconds)
        cmd_delay: Delay between commands (seconds)
    """
    # Initialize the coprocessor
    print("Initializing coprocessor...")
    board.init_coprocessor()
    time.sleep(0.1)
    
    # Send init sequence
    print("Sending init sequence...")
    board.send_nibble_stream(get_furby_command("init"))
    time.sleep(init_delay)
    
    # Run each command
    for cmd_name in command_names:
        print(f"\nExecuting: {cmd_name}")
        cmd = get_furby_command(cmd_name)
        if cmd:
            response = board.send_nibble_stream(cmd)
            print(f"  Sent {len(cmd)} hex chars")
            time.sleep(cmd_delay)
        else:
            print(f"  Warning: Command '{cmd_name}' not found")
    
    print("\nSequence complete!")

# Example: Run a sequence of commands
commands_to_run = ["mee-mee", "giggle", "yawn"]
run_furby_sequence(board, commands_to_run)

## Low-Level Command Interface

For advanced users, you can use the low-level `send_hex_command()` method to send raw control commands.

In [ ]:
# Low-level examples:

# Ping (control byte 0xFF)
response = board.send_hex_command("FF")
print("Ping:", response)

# Init coprocessor (control byte 0x01)
response = board.send_hex_command("01")
print("Init:", response)

# Send nibbles (control byte 0x02 + nibble data)
response = board.send_hex_command("02F7F347F7F71C")
print("Nibbles:", response[:100], "...")

## Summary

This notebook demonstrated:

1. Connecting to the Spirit Board
2. Testing connectivity with ping
3. Initializing the Furby coprocessor
4. Sending predefined command sequences
5. Parsing firmware responses
6. Controlling the motor
7. Creating custom command workflows

For more information, see the `spirit_board.py` source code and the project documentation.